# XGBoost-baseline en vergelijking met de SVM

**Datalab 3 · lesweek 12**

Dit notebook doet twee dingen:

1. **XGBoost-baseline:** een eerste XGBoost-model op de huidige 64 features, met standaardinstellingen. Geen tuning; dat komt in lesweek 13.
2. **Vergelijking met de SVM** uit `hommel_classificatie.ipynb`, op **precies dezelfde folds**: welk model is beter, en vooral bij `buzz`?

**Eerlijke evaluatie** (afspraken in `docs/planning.md`):

- We splitsen altijd **per opname** (`StratifiedGroupKFold` op `source_file`), met een vaste `random_state`.
- We gebruiken **dezelfde testset als `hommel_classificatie.ipynb`** (ca. 20% van de bestanden) en **raken die hier niet aan**. De testset gebruiken we één keer, in lesweek 16.
- Alle scores hieronder komen uit 5-voudige kruisvalidatie op de trainingsset.
- Hoofdmetric is **macro-F1**; daarnaast precision en recall per klasse. Voor het lectoraat is **precision buzz** extra belangrijk.

## 1. Instellingen

In [ ]:
import sys; print(sys.executable)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
KLASSEN = ["buzz", "flight", "ruis"]
FEATURES_PAD = "../data/processed_data/bee_features_per_second.csv"
TEKST_KOLOMMEN = ["start", "end", "label", "source_file"]

KLEUR_MODEL = {"XGBoost": "#2a78d6", "SVM": "#eb6834"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 3)

## 2. Data laden

Zelfde opschoning als `hommel_classificatie.ipynb`, zodat de folds gelijk zijn:

- ontdubbelen op `source_file` + `start` (elke rij stond er twee keer in);
- sorteren op bestand en tijd;
- klassen samenvoegen tot `buzz`, `flight` en `ruis` (= radio, voice en noise);
- alleen gelabelde seconden.

In [ ]:
features = pd.read_csv(FEATURES_PAD)
df = features.drop_duplicates(subset=["source_file", "start"]).copy()
df = df.sort_values(["source_file", "start"]).reset_index(drop=True)
print(f"Ingelezen: {len(features)} rijen, na ontdubbelen: {len(df)}")

df["klasse"] = df["label"].map({"buzz": "buzz", "flight": "flight"})
df.loc[df["label"].notna() & df["klasse"].isna(), "klasse"] = "ruis"

lab = df[df["klasse"].notna()].reset_index(drop=True)
FEATURES = [k for k in features.columns if k not in TEKST_KOLOMMEN]

X = lab[FEATURES]
y = lab["klasse"].values
groepen = lab["source_file"].values

print(f"Gelabelde seconden: {len(lab)} uit {lab['source_file'].nunique()} bestanden | features: {len(FEATURES)}")
pd.Series(y).value_counts().rename("seconden")

## 3. Testset apart zetten en folds maken

**Testset:** de eerste split van `StratifiedGroupKFold(5, shuffle=True, random_state=42)`, net als in `hommel_classificatie.ipynb`. Die zetten we opzij.

**Folds:** op de trainingsset maken we 5 folds met dezelfde splitter. We slaan de folds op in een lijst, zodat XGBoost en de SVM gegarandeerd op **precies dezelfde** seconden trainen en testen.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groepen))

X_train, y_train, g_train = X.iloc[train_idx].reset_index(drop=True), y[train_idx], groepen[train_idx]
assert set(g_train).isdisjoint(groepen[test_idx])  # geen bestand in train én test

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
FOLDS = list(cv.split(X_train, y_train, g_train))

for i, (tr, va) in enumerate(FOLDS, start=1):
    assert set(g_train[tr]).isdisjoint(g_train[va])
    telling = pd.Series(y_train[va]).value_counts()
    print(f"Fold {i}: {len(va):4d} seconden uit {len(set(g_train[va])):2d} bestanden | "
          + " | ".join(f"{k}: {telling.get(k, 0)}" for k in KLASSEN))

print(f"\nTestset opzij: {len(test_idx)} seconden uit {len(set(groepen[test_idx]))} bestanden (niet gebruikt)")

## 4. Evaluatiefunctie

Per fold: train op 4 delen, voorspel het vijfde deel, en bereken macro-F1 en precision en recall per klasse. `buzz` en `flight` zijn zeldzaam; daarom geven we beide modellen een **gewicht per klasse** (`balanced`): bij de SVM met `class_weight`, bij XGBoost met `sample_weight`.

In [ ]:
def evalueer(maak_model, gewichten=False):
    # Kruisvalidatie op FOLDS. Geeft (scores per fold, voorspellingen) terug.
    rijen = []
    voorspeld = np.empty(len(y_train), dtype=object)
    for i, (tr, va) in enumerate(FOLDS, start=1):
        model = maak_model()
        if gewichten:
            model.fit(X_train.iloc[tr], y_train[tr], sample_weight=compute_sample_weight("balanced", y_train[tr]))
        else:
            model.fit(X_train.iloc[tr], y_train[tr])
        pred = model.predict(X_train.iloc[va])
        voorspeld[va] = pred

        rij = {"fold": i, "macro-F1": f1_score(y_train[va], pred, average="macro", labels=KLASSEN, zero_division=0)}
        p = precision_score(y_train[va], pred, average=None, labels=KLASSEN, zero_division=0)
        r = recall_score(y_train[va], pred, average=None, labels=KLASSEN, zero_division=0)
        for k, pk, rk in zip(KLASSEN, p, r):
            rij[f"precision {k}"] = pk
            rij[f"recall {k}"] = rk
        rijen.append(rij)
    return pd.DataFrame(rijen).set_index("fold"), voorspeld


def met_gemiddelde(tabel):
    # Voegt rijen 'gemiddeld' en 'spreiding (sd)' toe.
    return pd.concat([tabel, tabel.agg(["mean", "std"]).rename(index={"mean": "gemiddeld", "std": "spreiding (sd)"})])

## 5. XGBoost-baseline

Standaardinstellingen van XGBoost, alleen een vaste `random_state`. XGBoost wil getallen als klassen; `XGBMetLabels` zet `buzz`/`flight`/`ruis` om naar 0/1/2 en terug.

In [ ]:
class XGBMetLabels(XGBClassifier):
    # XGBClassifier die tekstlabels accepteert.
    def fit(self, X, y, **kwargs):
        self.encoder_ = LabelEncoder().fit(y)
        return super().fit(X, self.encoder_.transform(y), **kwargs)

    def predict(self, X):
        return self.encoder_.inverse_transform(super().predict(X).astype(int))


def maak_xgb():
    return XGBMetLabels(random_state=SEED, n_jobs=-1, eval_metric="mlogloss")


scores_xgb, pred_xgb = evalueer(maak_xgb, gewichten=True)
met_gemiddelde(scores_xgb)

## 6. SVM op dezelfde folds

Dezelfde SVM als het gekozen model in `hommel_classificatie.ipynb`: `StandardScaler` + `SVC(class_weight="balanced")` met standaardinstellingen (`C=1`, `gamma="scale"`).

In [ ]:
def maak_svm():
    return make_pipeline(StandardScaler(), SVC(class_weight="balanced"))


scores_svm, pred_svm = evalueer(maak_svm)
met_gemiddelde(scores_svm)

## 7. Vergelijking

Gemiddelde en spreiding over de 5 folds. Omdat de folds gelijk zijn, kunnen we ook **per fold** vergelijken: wint XGBoost in elke fold, of alleen gemiddeld?

In [ ]:
METRICS = ["macro-F1", "precision buzz", "recall buzz", "precision flight", "recall flight", "precision ruis", "recall ruis"]

vergelijking = pd.DataFrame({
    "XGBoost gemiddeld": scores_xgb[METRICS].mean(),
    "XGBoost sd": scores_xgb[METRICS].std(),
    "SVM gemiddeld": scores_svm[METRICS].mean(),
    "SVM sd": scores_svm[METRICS].std(),
})
vergelijking["verschil (XGB − SVM)"] = vergelijking["XGBoost gemiddeld"] - vergelijking["SVM gemiddeld"]
vergelijking

In [ ]:
per_fold = pd.DataFrame({
    "macro-F1 XGBoost": scores_xgb["macro-F1"], "macro-F1 SVM": scores_svm["macro-F1"],
    "precision buzz XGBoost": scores_xgb["precision buzz"], "precision buzz SVM": scores_svm["precision buzz"],
})
per_fold["XGBoost beter (macro-F1)"] = per_fold["macro-F1 XGBoost"] > per_fold["macro-F1 SVM"]
per_fold

In [ ]:
fig, assen = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, metric in zip(assen, ["macro-F1", "precision buzz"]):
    for x, (naam, tabel) in enumerate([("XGBoost", scores_xgb), ("SVM", scores_svm)]):
        waarden = tabel[metric]
        ax.bar(x, waarden.mean(), yerr=waarden.std(), width=0.55, color=KLEUR_MODEL[naam],
               capsize=5, error_kw={"elinewidth": 1})
        ax.scatter(np.full(len(waarden), x) + np.linspace(-0.12, 0.12, len(waarden)), waarden,
                   s=18, color="#0b0b0b", zorder=3)
        ax.text(x, waarden.mean() + waarden.std() + 0.03, f"{waarden.mean():.2f}", ha="center", fontsize=10)
    # lijnen tussen dezelfde fold
    offsets = np.linspace(-0.12, 0.12, len(scores_xgb))
    for d, a, b in zip(offsets, scores_xgb[metric], scores_svm[metric]):
        ax.plot([0 + d, 1 + d], [a, b], color="#c3c2b7", linewidth=0.8, zorder=2)
    ax.set_xticks([0, 1], ["XGBoost", "SVM"])
    ax.set_title(metric, loc="left", fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.90, color="#898781", linestyle="--", linewidth=0.8)
    ax.text(1.32, 0.905, "norm 0,90", fontsize=8, color="#52514e", ha="right", va="bottom")
assen[0].set_ylabel("score (5 folds)")
fig.suptitle("XGBoost en SVM op dezelfde 5 folds: balk = gemiddelde, streep = spreiding, punt = fold",
             x=0.01, ha="left", fontsize=10)
plt.tight_layout()
plt.show()

**Verwarringen.** Welke klassen haalt elk model door elkaar? Opgeteld over de 5 folds (rijen = echt, kolommen = voorspeld).

In [ ]:
for naam, pred in [("XGBoost", pred_xgb), ("SVM", pred_svm)]:
    print(naam)
    print(pd.crosstab(pd.Series(y_train, name="echt"), pd.Series(pred, name="voorspeld")).reindex(index=KLASSEN, columns=KLASSEN, fill_value=0))
    print()

## 8. Conclusie

De cel hieronder zet de belangrijkste cijfers op een rij. Een verschil dat kleiner is dan de spreiding over de folds noemen we **niet duidelijk**.

In [ ]:
def samenvatting(metric):
    a, b = scores_xgb[metric], scores_svm[metric]
    verschil = a.mean() - b.mean()
    spreiding = max(a.std(), b.std())
    wint = (a > b).sum()
    if abs(verschil) <= spreiding:
        oordeel = "geen duidelijk verschil (kleiner dan de spreiding)"
    else:
        oordeel = "XGBoost duidelijk beter" if verschil > 0 else "SVM duidelijk beter"
    return (f"{metric}: XGBoost {a.mean():.2f} (sd {a.std():.2f}), SVM {b.mean():.2f} (sd {b.std():.2f}); "
            f"XGBoost wint in {wint} van de 5 folds -> {oordeel}")


for m in ["macro-F1", "precision buzz", "recall buzz"]:
    print(samenvatting(m))

**Conclusie voor Sam**

- **Macro-F1:** de SVM scoort gemiddeld hoger dan XGBoost (0,57 tegen 0,47) en wint in 4 van de 5 folds. Het verschil is wel kleiner dan de spreiding tussen de folds, dus nog niet hard.
- **Buzz:** beide modellen halen een precision buzz van ongeveer 0,4, met heel veel spreiding (van 0 tot 1 per fold). XGBoost voorspelt bijna nooit `buzz`: het vindt maar 15% van de buzz-seconden, de SVM 44%.
- **Verwarringen:** XGBoost zet de meeste hommelseconden op `ruis`. De SVM vindt meer hommelgeluid, maar geeft ook vaker vals alarm (164 ruis-seconden als buzz of flight). `buzz` en `flight` worden door beide modellen vaak met elkaar verward.
- **Normen:** geen van beide modellen haalt de norm van 0,90.
- **Vervolg:** in lesweek 13 tunen we XGBoost met een grid search op dezelfde folds, en komen er nieuwe features uit de audio (dominante frequentie, bandenergie). Daarna draaien we de vergelijking opnieuw op de gerepareerde features.